# 语义分割算法配置文件构建 - UNet (自定义数据集)

本模块负责加载官方预训练模型的基础配置，替换为自定义的数据集路径与处理流水线，修改分类头参数，并导出最终的训练配置文件。

## 1. 环境准备与工作目录设定

本节负责导入核心配置引擎库，并锁定工作路径至 `mmsegmentation` 根目录。同时建立专属的配置保存文件夹。

In [1]:
import os
from mmengine import Config

# 1. 锁定绝对工作目录
WORK_DIR = '/root/LearningMMSegmentation/mmsegmentation'
os.chdir(WORK_DIR)

# 2. 创建自定义配置文件的专属保存目录
SAVE_DIR = 'My-Configs'
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"✅ 环境初始化完成。当前工作目录: {os.getcwd()}")
print(f"📁 配置文件将保存至: {os.path.join(WORK_DIR, SAVE_DIR)}")

✅ 环境初始化完成。当前工作目录: /root/LearningMMSegmentation/mmsegmentation
📁 配置文件将保存至: /root/LearningMMSegmentation/mmsegmentation/My-Configs


## 2. 加载基础配置与模型结构调整

加载 UNet 算法在 Cityscapes 数据集上的基础配置文件。针对单 GPU 训练环境，将分布式同步批归一化（SyncBN）退化为标准批归一化（BN）。同时，根据自定义数据集的实际情况修改分类头的类别数量。

In [2]:
# 1. 载入基础模型 config 配置文件
base_config_path = './configs/unet/unet-s5-d16_fcn_4xb4-160k_cityscapes-512x1024.py'
cfg = Config.fromfile(base_config_path)

# 2. 统一归一化层配置 (单卡环境需使用 BN 而非 SyncBN)
norm_cfg = dict(type='BN', requires_grad=True)
cfg.norm_cfg = norm_cfg
cfg.model.backbone.norm_cfg = norm_cfg
cfg.model.decode_head.norm_cfg = norm_cfg
cfg.model.auxiliary_head.norm_cfg = norm_cfg

# 3. 修改分类输出的类别数
# 【重点参数】请修改 NUM_CLASSES 为自定义数据集的真实类别数（包含背景类）
NUM_CLASSES = 6 
cfg.model.decode_head.num_classes = NUM_CLASSES
cfg.model.auxiliary_head.num_classes = NUM_CLASSES

print(f"✅ 模型结构参数修改完成。当前设定类别数为: {NUM_CLASSES}")

✅ 模型结构参数修改完成。当前设定类别数为: 6


## 3. 数据集与数据流水线 (Pipeline) 配置

将基础配置中的数据集替换为步骤 E 中准备好的自定义数据集。设定数据根目录，并将数据流水线挂载至训练（Train）、验证（Val）和测试（Test）的数据加载器中。

In [3]:
# 1. 强制修改数据集类型与数据根目录
DATASET_TYPE = 'ZihaoDataset' 
DATA_ROOT = 'Watermelon87_Semantic_Seg_Mask/' 

cfg.dataset_type = DATASET_TYPE
cfg.data_root = DATA_ROOT

# ========================================================
# 🌟 终极修正：还原原尺寸评测机制的验证流水线
# ========================================================
test_pipeline = [
    dict(type='LoadImageFromFile'),
    dict(type='Resize', scale=(2048, 1024), keep_ratio=True),
    # 修正 1：只对 img 进行 Pad 保护 UNet，去掉 gt_seg_map，严禁修改标签尺寸！
    dict(type='Pad', size_divisor=16, pad_val=dict(img=0)),
    # 修正 2：在 Pad 之后加载原始标注图，保证评测时 ground truth 保持原始尺寸
    dict(type='LoadAnnotations'),
    dict(type='PackSegInputs')
]

# 覆盖训练集 Pipeline 路径
cfg.train_dataloader.dataset.type = DATASET_TYPE
cfg.train_dataloader.dataset.data_root = DATA_ROOT
cfg.train_dataloader.dataset.data_prefix = dict(img_path='img_dir/train', seg_map_path='ann_dir/train')

# 覆盖验证集 Pipeline
cfg.val_dataloader.dataset.type = DATASET_TYPE
cfg.val_dataloader.dataset.data_root = DATA_ROOT
cfg.val_dataloader.dataset.data_prefix = dict(img_path='img_dir/val', seg_map_path='ann_dir/val')
cfg.val_dataloader.dataset.pipeline = test_pipeline

# 清理可能残留的冲突属性
if hasattr(cfg.val_dataloader.dataset, 'img_suffix'):
    cfg.val_dataloader.dataset.pop('img_suffix')
if hasattr(cfg.val_dataloader.dataset, 'seg_map_suffix'):
    cfg.val_dataloader.dataset.pop('seg_map_suffix')

# 覆盖测试集 Pipeline
cfg.test_dataloader = cfg.val_dataloader

print(f"✅ 数据集与 Pipeline 配置完成。已正确隔离输入图像与标注图的预处理逻辑。")

✅ 数据集与 Pipeline 配置完成。已正确隔离输入图像与标注图的预处理逻辑。


## 4. 训练策略与超参数设定

配置总训练迭代次数、验证频率、学习率调度器（Scheduler）以及检查点（Checkpoint）保存策略。

In [4]:
# 1. 设定训练核心周期参数
MAX_ITERS = 2500       # 总训练迭代次数
VAL_INTERVAL = 250      # 验证及评估间隔（每 N 步进行一次测试集评估）
LOG_INTERVAL = 100      # 日志打印间隔

cfg.train_cfg.max_iters = MAX_ITERS
cfg.train_cfg.val_interval = VAL_INTERVAL
cfg.default_hooks.logger.interval = LOG_INTERVAL

# 2. 设定评估指标
cfg.val_evaluator.iou_metrics = ['mIoU', 'mDice', 'mFscore']
cfg.test_evaluator = cfg.val_evaluator

# 3. 设定模型权重保存策略
cfg.default_hooks.checkpoint = dict(
    type='CheckpointHook',
    by_epoch=False,
    interval=250,        # 每 500 步保存一次权重文件
    max_keep_ckpts=1,     # 仅保留最新的 1 个权重文件，防止磁盘占用过大
    save_best='mIoU'      # 自动保存验证集 mIoU 最高的权重为 best_model.pth
)

# 4. 设定学习率与优化器
cfg.optim_wrapper.optimizer.lr = 0.01          # 初始学习率
cfg.param_scheduler = [
    dict(
        type='PolyLR',
        eta_min=1e-4,
        power=0.9,
        begin=0,
        end=MAX_ITERS,
        by_epoch=False)
]

# 5. 其他配置
cfg.resume = False            # 是否从断点恢复训练
cfg.randomness = dict(seed=0) # 固定随机数种子，确保实验可复现

print(f"✅ 训练策略配置完成。总迭代次数: {MAX_ITERS}, 验证间隔: {VAL_INTERVAL}")

✅ 训练策略配置完成。总迭代次数: 2500, 验证间隔: 250


## 5. 导出最终配置文件

对修改后的全局 Config 对象进行序列化，并导出保存至指定目录，供后续调用 `tools/train.py` 脚本执行训练时读取。

In [5]:
# 定义最终配置文件的保存路径
final_config_path = os.path.join(SAVE_DIR, 'CustomDataset_UNet_pipeline.py')

# 导出配置文件
cfg.dump(final_config_path)

print(f"🎉 完美！专属的 UNet 训练配置文件已成功导出保存至:")
print(f"👉 {os.path.abspath(final_config_path)}")
print("\n后续可在终端或 Notebook 中通过以下指令启动训练：")
print(f"!python tools/train.py {final_config_path}")

🎉 完美！专属的 UNet 训练配置文件已成功导出保存至:
👉 /root/LearningMMSegmentation/mmsegmentation/My-Configs/CustomDataset_UNet_pipeline.py

后续可在终端或 Notebook 中通过以下指令启动训练：
!python tools/train.py My-Configs/CustomDataset_UNet_pipeline.py
